# 02 資料處理與視覺化 — 參考解答

松柏護理之家退伍軍人症 line list 練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import plotly.express as px
import plotly.io as pio

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# Plotly: 確保在靜態建置（jupyter-book build）時也能輸出互動圖
pio.renderers.default = "notebook"


## 題目 1：讀入並檢視資料

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
print(f"資料維度：{df.shape[0]} 筆 × {df.shape[1]} 欄")
print(f"\n欄位名稱：{df.columns.tolist()}")
df.head()

In [ ]:
df.info()

## 題目 2：日期轉換與衍生變項

In [ ]:
# 日期轉換
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")

# 建立 infected 欄位
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 計算 onset_to_hosp_days
df["onset_to_hosp_days"] = (
    df["hospitalization_date"] - df["symptom_onset_date"]
).dt.days

# 印出前 10 位感染者
infected_df = df[df["infected"] == 1]
infected_df[["case_id", "symptom_onset_date", "onset_to_hosp_days"]].head(10)

## 題目 3：流行曲線

In [ ]:
import matplotlib.dates as mdates

cases = df[df["infected"] == 1]
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# 加入爆發前背景期（含零病例日）
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
)
daily = daily.reindex(date_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    daily.index, daily.values,
    width=1.0,
    color="#2c7fb8", edgecolor="white", linewidth=0.5,
)
ax.set_title(
    "松柏護理之家退伍軍人症流行曲線，依發病日，2026 年 1 月",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 題目 4：翼區侵襲率比較圖

In [ ]:
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate_pct"] = (
    wing_stats["infected"] / wing_stats["residents"] * 100
).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]
wing_stats = wing_stats.sort_values("attack_rate_pct", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=wing_stats, x="label", y="attack_rate_pct",
    hue="label", palette="YlOrRd", legend=False, ax=ax,
)
ax.set_title("各翼區侵襲率比較")
ax.set_xlabel("翼區")
ax.set_ylabel("侵襲率 (%)")

for i, row in enumerate(wing_stats.itertuples()):
    ax.text(i, row.attack_rate_pct + 1, f"{row.attack_rate_pct}%",
            ha="center", fontsize=10)

plt.tight_layout()
plt.show()

## 題目 5（挑戰題）：互動式分層流行曲線

In [ ]:
daily_floor = (
    cases.groupby(["symptom_onset_date", "floor"])
    .size()
    .rename("cases")
    .reset_index()
)
daily_floor["floor"] = daily_floor["floor"].astype(str) + "F"

fig = px.bar(
    daily_floor,
    x="symptom_onset_date", y="cases", color="floor",
    barmode="stack",
    title="松柏護理之家退伍軍人症流行曲線（依樓層分層），2026 年 1 月",
    labels={"symptom_onset_date": "發病日期", "cases": "病例數", "floor": "樓層"},
)
fig.update_layout(
    bargap=0,
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=False, rangemode="tozero"),
    plot_bgcolor="white",
)
fig.show()

### 解讀

- **流行曲線**：病例高峰集中在數天之內，呈現典型的 **共同暴露源（point source）** 型態
- **翼區比較**：3B 翼侵襲率最高，1B 翼最低 → 暴露源可能與特定區域設施有關
- **分層曲線**：若三樓流行高峰早於一樓，可能暗示暴露源在高樓層（例如水塔供水管路）

## 題目 6：頻率表與樞紐分析

In [ ]:
# 嚴重度次數分布
print("=== 嚴重度次數分布 ===")
print(df["clinical_severity"].value_counts())
print("\n=== 嚴重度百分比 ===")
print(df["clinical_severity"].value_counts(normalize=True).mul(100).round(1))

# 翼區 × 樓層 侵襲率表
print("\n=== 翼區 × 樓層 侵襲率 ===")
pivot = pd.pivot_table(
    df,
    values="infected",
    index="wing",
    columns="floor",
    aggfunc="mean",
    margins=True,
)
print(pivot.round(3))

## 題目 7：Method Chaining

In [ ]:
# 用 method chaining 一行完成：篩選 70+ 感染者 → 按樓層分組 → 算人數和死亡數 → 致死率 → 排序
result = (
    df
    .query("infected == 1 and age >= 70")
    .groupby("floor")
    .agg(
        n_cases=("case_id", "count"),
        n_deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .assign(cfr=lambda d: (d["n_deaths"] / d["n_cases"] * 100).round(1))
    .sort_values("cfr", ascending=False)
)
print("70 歲以上感染者，各樓層致死率：")
result

## 題目 8：合併資料與文字清理

In [ ]:
import numpy as np

# 1. 模擬實驗室檢驗資料
rng = np.random.default_rng(42)
infected_ids = df.loc[df["infected"] == 1, "case_id"].tolist()
lab = pd.DataFrame({
    "case_id": infected_ids,
    "ct_value": rng.uniform(15, 35, size=len(infected_ids)).round(1),
})
print(f"Lab 資料：{len(lab)} 筆")
lab.head()

In [ ]:
# 2. 合併 lab 資料
df_merged = pd.merge(df, lab, on="case_id", how="left")
print(f"合併後：{df_merged.shape}")
print(f"有 ct_value 的筆數：{df_merged['ct_value'].notna().sum()}")

# 3. 統一 wing 大小寫
df_merged["wing"] = df_merged["wing"].str.strip().str.upper()
print(f"\nWing 類別：{df_merged['wing'].unique()}")

# 4. 去除重複通報
before = len(df_merged)
df_merged = df_merged.drop_duplicates(subset="case_id", keep="first")
print(f"去重前：{before}，去重後：{len(df_merged)}")

# 5. 年齡最大的 5 位感染者
top5 = df_merged.query("infected == 1").nlargest(5, "age")
print("\n年齡最大的 5 位感染者：")
top5[["case_id", "age", "floor", "wing", "clinical_severity", "ct_value"]]